## Inference Phase for one Tile

In [4]:
import rasterio
import numpy as np
import os
from osgeo import gdal
import joblib
import time
import pandas as pd
from tqdm import tqdm

### Load the Tif File

### Create an empty tif file, to save the predictions

In [5]:
def create_empty_tiff(output_path, bands=39, rows=5000, cols=5000):
    """Creates an empty GeoTIFF file with specified dimensions."""
    driver = gdal.GetDriverByName('GTiff')
    
    # Delete output file if it exists
    if os.path.exists(output_path):
        os.remove(output_path)
    
    # Create new file
    dataset = driver.Create(output_path, 
                        cols,        # width
                        rows,        # height
                        bands,       # number of bands
                        gdal.GDT_Float32)  # data type
    
    # Initialize with zeros if needed
    for i in range(bands):
        band = dataset.GetRasterBand(i + 1)
        band.Fill(0)  # Fill with zeros
        band.FlushCache()
    
    # If you need to set geotransform and projection, you can copy from existing file:
    example_file = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/NDVI/2023_stacked_NDVI_NBR.tif'
    with rasterio.open(example_file) as src:
        dataset.SetGeoTransform(src.transform.to_gdal())
        dataset.SetProjection(src.crs.to_wkt())
    
    dataset.FlushCache()
    return dataset


output_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/RF_predictions.tif'
empty_dataset = create_empty_tiff(output_path)
print(f"Created empty TIFF file at: {output_path}")

/usr/lib/python3/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Created empty TIFF file at: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/RF_predictions.tif


### Load the RF Model

In [6]:

# Load the trained Random Forest model
model_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/clean_notebooks2/random_forest_model.pkl'
rf_model = joblib.load(model_path)
print(f"Loaded Random Forest model from: {model_path}")



Loaded Random Forest model from: /home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/clean_notebooks2/random_forest_model.pkl


### Load the reshaped tile

In [7]:
# Load the reshaped tile data
reshaped_tile_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/tiles/reshaped_tile_for_inference.npz'
reshaped_tile = np.load(reshaped_tile_path)
X_tile = reshaped_tile['data']  
print(X_tile.shape)


(975000000, 4)


### Make the predictions

In [8]:
# Define column names - use the same names as in training
column_names = ['NDVI', 'NDVI_diff', 'NBR', 'NBR_diff']  # adjust these to match your training data

# Convert X_tile to DataFrame
X_tile_df = pd.DataFrame(X_tile, columns=column_names)

# Now make predictions in batches
batch_size = 1000000
num_batches = len(X_tile_df) // batch_size + (1 if len(X_tile_df) % batch_size != 0 else 0)

# Initialize empty array for predictions
predictions = np.zeros((39, 5000, 5000), dtype=np.float32)

# Process in batches
for batch_idx in tqdm(range(num_batches)):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(X_tile_df))
    
    # Get batch as DataFrame
    batch_df = X_tile_df.iloc[start_idx:end_idx]
    
    # Make predictions for this batch
    batch_predictions = rf_model.predict(batch_df)
    
    # Calculate corresponding indices in the final image (!! nach pixeln)
    pixel_indices = np.arange(start_idx, end_idx)
    time_indices = pixel_indices // (5000 * 5000)
    spatial_indices = pixel_indices % (5000 * 5000)
    row_indices = spatial_indices // 5000
    col_indices = spatial_indices % 5000
    
    # Place predictions in the correct locations
    predictions[time_indices, row_indices, col_indices] = batch_predictions
    
# Write predictions to the empty TIF file
print("\nWriting predictions to file...")
for i in range(39):
    band = empty_dataset.GetRasterBand(i + 1)
    band.WriteArray(predictions[i])
    band.FlushCache()

# Close the dataset
empty_dataset = None

total_time = time.time() - start_time
print(f"\nTotal processing time: {total_time/3600:.2f} hours")

100%|██████████| 975/975 [09:09<00:00,  1.77it/s]



Writing predictions to file...


NameError: name 'start_time' is not defined

In [ ]:
# Define column names - use the same names as in training
column_names = ['NDVI', 'NDVI_diff', 'NBR', 'NBR_diff']  # adjust these to match your training data

# Initialize empty array for predictions
predictions = np.zeros((39, 5000, 5000), dtype=np.float32)

# Start timing
start_time = time.time()

# Process row by row, column by column
for row in tqdm(range(5000), desc="Processing rows"):
    for col in range(5000):
        # Get all time steps for this pixel
        pixel_data = X_tile[:, row * 5000 + col]
        
        # Convert to DataFrame for prediction
        pixel_df = pd.DataFrame(pixel_data.reshape(-1, 4), columns=column_names)
        
        # Make predictions for all time steps at once
        predictions[:, row, col] = rf_model.predict(pixel_df)

# Write predictions to the empty TIF file
print("\nWriting predictions to file...")
for i in range(39):
    band = empty_dataset.GetRasterBand(i + 1)
    band.WriteArray(predictions[i])
    band.FlushCache()

# Close the dataset
empty_dataset = None

total_time = time.time() - start_time
print(f"\nTotal processing time: {total_time/3600:.2f} hours")